In [ ]:
from tqdm.auto import tqdm
from pdf2image import convert_from_path

# Path to your PDF
pdf_path = "../data/KR Notation Guide_2025.pdf"
filename = pdf_path.split("/")[-1].replace(".pdf", "")

# Convert all pages to images
images = convert_from_path(pdf_path, dpi=300)

# Save each page as a separate image
for i, page in tqdm(enumerate(images)):
    image_path = f"../data/notation/{filename}_{i}.png"
    page.save(image_path, "PNG")
    print(f"Saved: {image_path}")


In [12]:
import os
import pickle
from time import time

import pandas as pd
from langchain_core.documents import Document

from docling.document_converter import DocumentConverter, ImageFormatOption
from docling.models.tesseract_ocr_cli_model import TesseractCliOcrOptions



file_path = "../data/notation/KR Notation Guide_2025_75.png"
lv1_cat, lv2_cat = "Rule", "KR"

path = file_path.replace("\\", "/")
filename = path.split("/")[-1].replace(".png", "").replace(".jpg", "")
first_sentence = f"This page explains {filename.replace(".pdf", "")} that belongs to {lv1_cat} and  {lv2_cat} categories.\n"


# Configure OCR for image input
image_options = ImageFormatOption(
    ocr_options=TesseractCliOcrOptions(force_full_page_ocr=True),
    do_table_structure=True,
    table_structure_options={"do_cell_matching": True},
)

converter = DocumentConverter(
    format_options={"image": image_options}
)

start_time = time()
conv_res = converter.convert(file_path).document

tables = []
# Print all tables as Markdown
for table_ix, table in enumerate(conv_res.tables):
    table_df: pd.DataFrame = table.export_to_dataframe(doc=conv_res)
    page_num = table.prov[0].page_no if table.prov else "Unknown"
    extracted_table = first_sentence + table_df.to_markdown()
    lang_table = Document(page_content=extracted_table, metadata={'filename': filename, 'lv1_cat': lv1_cat, 'lv2_cat': lv2_cat, 'page':str(page_num)})
    tables.append(lang_table)
    
parsed_foldername = f"{lv1_cat}_{lv2_cat}"
if not os.path.exists(f"../docs/{parsed_foldername}_img/notation"):
    os.makedirs(f"../docs/{parsed_foldername}_img/notation")

parsed_filename = filename.replace(".png", "").replace(".jpg", "")
with open(f"../docs/{parsed_foldername}_img/notation/{parsed_filename}.pkl", 'ab') as file:
    pickle.dump(tables, file)
        

end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

2025-10-08 10:44:44,170 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-10-08 10:44:44,260 - INFO - Going to convert document batch...
2025-10-08 10:44:44,261 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-08 10:44:44,263 - INFO - Accelerator device: 'cpu'
2025-10-08 10:44:46,437 - INFO - Accelerator device: 'cpu'
2025-10-08 10:44:48,054 - INFO - Accelerator device: 'cpu'
2025-10-08 10:44:48,833 - INFO - Processing document KR Notation Guide_2025_75.png
d:\my_parser\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2025-10-08 10:45:34,442 - INFO - Finished converting document KR Notation Guide_2025_75.png in 50.30 sec.


Document converted and tables exported in 50.31 seconds.


In [13]:
import pickle

file_path = "../docs/Rule_KR_img/notation/KR Notation Guide_2025_75.pkl"
# file_path = "../docs/Rule_KR_img/page_1.png.pkl"
# Open the file containing the pickled data in binary read mode ('rb')
with open(file_path, 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [14]:
loaded_object

[Document(metadata={'filename': 'KR Notation Guide_2025_75', 'lv1_cat': 'Rule', 'lv2_cat': 'KR', 'page': '1'}, page_content="This page explains KR Notation Guide_2025_75 that belongs to Rule and  KR categories.\n|    | Ship Type Notations.                                                  | Special Feature Notations.OilTanker                  | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier                           | Special Feature Notations.Liquefied Gas Carrier   |\n|---:|:----------------------------------------------------------------------|:-----------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------------------------------|:---

In [15]:
from IPython.display import Markdown
Markdown(loaded_object[0].page_content)

This page explains KR Notation Guide_2025_75 that belongs to Rule and  KR categories.
|    | Ship Type Notations.                                                  | Special Feature Notations.OilTanker                  | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier   | Special Feature Notations.Liquefied Gas Carrier                           | Special Feature Notations.Liquefied Gas Carrier   |
|---:|:----------------------------------------------------------------------|:-----------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------|:--------------------------------------------------------------------------|:--------------------------------------------------|
|  0 | Oil/Liquefied Gas Carrier 'ESP' (Double Hull) (FAC) (FAO) (FBC) (CSR) | Crude Product Crude /Product Product/Asphalt Asphalt | A                                                 | B                                                 | (C)                                               | Design Aspect and/or Primary Carg                                         | IMO Code                                          |
|  1 | (Double Hull)(EXP)                                                    |                                                      | 1G 2G                                             | 2/                                                |                                                   | Maximum  Vapour Pressure; Minimum  Temperature and Specific  Gravity (SG) | (NIGC) (IGC) (GC) (GCX)                           |
|  2 |                                                                       |                                                      |                                                   | 3M                                                |                                                   |                                                                           |                                                   |
|  3 |                                                                       |                                                      | 2PG                                               | 3S                                                |                                                   |                                                                           |                                                   |
|  4 |                                                                       |                                                      |                                                   |                                                   | (RP)                                              |                                                                           |                                                   |
|  5 |                                                                       |                                                      | 3G                                                |                                                   |                                                   |                                                                           |                                                   |
|  6 |                                                                       |                                                      |                                                   | 1A                                                |                                                   |                                                                           |                                                   |
|  7 |                                                                       |                                                      |                                                   | 1B                                                |                                                   | Name of Liquefied Gas                                                     |                                                   |
|  8 |                                                                       |                                                      |                                                   | 1C                                                |                                                   |                                                                           |                                                   |
|  9 |                                                                       |                                                      |                                                   | NV                                                |                                                   | primarily   carried                                                       |                                                   |